In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))

from src.soja_clima.merge import consolidar_base_final

PROCESSED_DIR = Path("../../data/processed")

In [2]:
df_clima = pd.read_parquet(PROCESSED_DIR / "clima_diario.parquet")
df_soja = pd.read_parquet(PROCESSED_DIR / "soja_microrregiao.parquet")
df_solo = pd.read_parquet(PROCESSED_DIR / "solo_microrregioes_limpo.parquet")
df_pacifico = pd.read_parquet(PROCESSED_DIR / "pacifico_indices.parquet")
df_mapa = pd.read_csv(PROCESSED_DIR / "estacao_microrregiao.csv") # <-- PRECISA DESSA LINHA

print("Arquivos carregados!")

Arquivos carregados!


In [3]:
df_final = consolidar_base_final(df_clima, df_soja, df_solo, df_pacifico, df_mapa)

Padronizando chaves de integração...
  - mapa: removidas 2 linhas com CD_MICRO nulo
Cruzando Clima com o Mapa de Microrregiões...
Cruzando Clima e Pacífico...
Agregando clima diário para anual...
Cruzando Soja e Clima...
Cruzando com Solo (estático)...
Base final consolidada com 6998 linhas e 40 colunas.


In [6]:
print("="*60)
print(" DIAGNÓSTICO REAL (NÍVEL DE LINHA E COBERTURA)")
print("="*60)

# 1. Qual a proporção de linhas com clima completo?
total = len(df_final)
nulos_clima = df_final['precipitacao_total'].isnull().sum()
completas = total - nulos_clima
print(f"Total de linhas na base: {total}")
print(f"Linhas com clima completo: {completas} ({completas/total:.1%})")
print(f"Linhas com clima nulo: {nulos_clima} ({nulos_clima/total:.1%})")

# 2. Quantas microrregiões têm PELO MENOS UM ano com clima?
micro_com_algum_clima = df_final.dropna(subset=['precipitacao_total'])['CD_MICRO'].nunique()
print(f"\nMicrorregiões com pelo menos 1 ano de clima: {micro_com_algum_clima} de 344")

# 3. Quantas microrregiões NÃO têm NENHUM dado de clima?
micro_sem_nada = df_final.dropna(subset=['precipitacao_total'])['CD_MICRO'].nunique()
micro_sem_nada = 344 - micro_sem_nada
print(f"Microrregiões sem NENHUM dado de clima: {micro_sem_nada}")

# 4. Distribuição de cobertura (quantos anos cada microrregião tem)
cobertura = df_final.groupby('CD_MICRO').agg(
    anos_totais=('ano', 'nunique'),
    anos_com_clima=('precipitacao_total', lambda x: x.notna().sum())
)
cobertura['pct_cobertura'] = (cobertura['anos_com_clima'] / cobertura['anos_totais'] * 100).round(1)

print(f"\nDistribuição de cobertura temporal por microrregião:")
print(f"  - 100% dos anos com clima: {(cobertura['pct_cobertura'] == 100).sum()} microrregiões")
print(f"  - 80-99% dos anos com clima: {((cobertura['pct_cobertura'] >= 80) & (cobertura['pct_cobertura'] < 100)).sum()} microrregiões")
print(f"  - 50-79% dos anos com clima: {((cobertura['pct_cobertura'] >= 50) & (cobertura['pct_cobertura'] < 80)).sum()} microrregiões")
print(f"  - 1-49% dos anos com clima: {((cobertura['pct_cobertura'] >= 1) & (cobertura['pct_cobertura'] < 50)).sum()} microrregiões")
print(f"  - 0% (nunca teve clima): {(cobertura['pct_cobertura'] == 0).sum()} microrregiões")

 DIAGNÓSTICO REAL (NÍVEL DE LINHA E COBERTURA)
Total de linhas na base: 6998
Linhas com clima completo: 4782 (68.3%)
Linhas com clima nulo: 2216 (31.7%)

Microrregiões com pelo menos 1 ano de clima: 327 de 344
Microrregiões sem NENHUM dado de clima: 17

Distribuição de cobertura temporal por microrregião:
  - 100% dos anos com clima: 41 microrregiões
  - 80-99% dos anos com clima: 66 microrregiões
  - 50-79% dos anos com clima: 192 microrregiões
  - 1-49% dos anos com clima: 28 microrregiões
  - 0% (nunca teve clima): 17 microrregiões


In [5]:
df_final.to_csv(PROCESSED_DIR / "base_final_modelagem.csv", index=False)
print(f"\n✅ Base final exportada para data/processed/base_final_modelagem.csv")


✅ Base final exportada para data/processed/base_final_modelagem.csv
